# dino_sim Colab trainer

Runs **in parallel** with the local RTX 5080 job. Same C++ island + PPO, different seed and checkpoint directory so the two runs do not overwrite each other.

1. Runtime → Change runtime type → **GPU**
2. Run all cells. This clones `https://github.com/rao30/dino_sim` and writes checkpoints to Drive `dino_sim/runs/colab/` so a disconnect does not lose the run.

The island sim is CPU; Colab’s GPU only does PPO. Use fewer worlds than the 5080 (default 32).

In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path

from google.colab import drive
drive.mount("/content/drive")

DRIVE = Path("/content/drive/MyDrive")
REPO = Path("/content/dino_sim")
RUN_DIR = DRIVE / "dino_sim" / "runs" / "colab"
RUN_DIR.mkdir(parents=True, exist_ok=True)
REPO_URL = os.environ.get("DINO_SIM_GIT", "https://github.com/rao30/dino_sim.git")

if REPO.exists():
    shutil.rmtree(REPO)
subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(REPO)])
print("repo", REPO, "has CMakeLists", (REPO / "CMakeLists.txt").exists())

In [ ]:
import torch
print("cuda", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
assert torch.cuda.is_available(), "Runtime → Change runtime type → GPU"
os.chdir(REPO)
subprocess.check_call(["bash", "scripts/colab_setup.sh"])

In [ ]:
# Do not pip-install train/requirements.txt torch (that wheel is for the 5080 / cu130).
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "numpy"])
os.environ["PYTHONUNBUFFERED"] = "1"
os.environ["PYTHONPATH"] = str(REPO / "train")

cmd = [
    sys.executable, "train/train_phase1.py",
    "--run-dir", str(RUN_DIR),
    "--seed", "7",
    "--worlds", "32",
    "--raptors", "12",
    "--gallis", "20",
    "--arena", "96",
    "--rollout", "256",
    "--bc-iters", "60",
    "--eval-every", "20",
    "--device", "cuda",
]
print(" ".join(cmd), flush=True)
subprocess.check_call(cmd)

After this cell finishes (or if you interrupt), eval the Colab checkpoint without touching the local 5080 run:

```
python train/train_phase1.py --eval --run-dir /content/drive/MyDrive/dino_sim/runs/colab --device cuda
```

Copy `raptor_best.pt` from Drive back to the Windows box and compare with `train/runs/raptor_latest.pt`.